In [4]:
# INSTALASI ULTRALYTICS & IMPORT LIBARIES

!pip install ultralytics
from ultralytics import YOLO
import sys
import torch

In [5]:
# CEK VERSI PYTHON, TORCH, CUDA, DAN KETERSEDIAAN GPU

print("Python:", sys.executable)
print("Torch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("Using device:", "cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

# Apabila tidak tersedia GPU, maka akan menggunakan CPU untuk pelatihan model.

Python: e:\Vscode Projects\yolov11-project-final\venv\Scripts\python.exe
Torch: 2.5.1+cu121
CUDA build: 12.1
CUDA available: True
Using device: cuda
GPU name: NVIDIA GeForce GTX 1660 SUPER


In [6]:
# LOAD MODEL YOLOv11N

model = YOLO("yolo11n.pt")

In [ ]:
# TRAINING MODEL YOLOv11N DENGAN HYPERPARAMETER TUNING UNTUK PRECISION DAN EVALUASI HASIL TRAINING
# Menggunakan hyperparameter tuning yang cocok dengan GPU GTX 1660 Super

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model.train(
    data="data.yaml",
    epochs=100,
    imgsz=640,
    batch=4,
    device=device,
    workers=2,
    cache=False,
    amp=True,

    # 🔥 tuning untuk precision
    lr0=0.002,          # lebih kecil → training lebih hati-hati
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0007,

    box=7.5,            # fokus bounding box
    cls=0.5,            # kurangi over-classification
    dfl=1.5,

    patience=30,
    cos_lr=True,
    save=True
)

# evaluasi
metrics = model.val(device=device)

print("===== HASIL TRAINING =====")
print(f"Precision : {metrics.box.mp:.4f}")
print(f"Recall    : {metrics.box.mr:.4f}")
print(f"mAP50     : {metrics.box.map50:.4f}")
print(f"mAP50-95  : {metrics.box.map:.4f}")

Ultralytics 8.4.41  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1660 SUPER, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=30, pers

In [9]:
# TESTING MODEL PADA IMAGE DAN VIDEO

from ultralytics import YOLO
import os
import sys

VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv", ".webm"}

def load_model(model_path):
    """Load YOLO model from specified path."""
    if not os.path.exists(model_path):
        raise FileNotFoundError("❌ Model tidak ditemukan!")
    return YOLO(model_path)

def get_user_choice():
    """Display menu and get user choice."""
    menu_text = (
        "\n=== OBJECT DETECTION SYSTEM ===\n"
        "1. Image Detection\n"
        "2. Video Tracking\n"
    )
    print(menu_text)
    sys.stdout.flush()
    choice = input("Pilih menu (1/2): ").strip()
    return choice

def get_file_path():
    """Get and validate file path from user."""
    source = input("Masukkan path file: ").strip()
    if not os.path.exists(source):
        raise FileNotFoundError("❌ File tidak ditemukan!")
    return source

def is_video_file(path):
    """Check whether the source path points to a video file."""
    _, ext = os.path.splitext(path)
    return ext.lower() in VIDEO_EXTENSIONS

def run_image_detection(model, source):
    """Run image detection."""
    print("\n[MODE] IMAGE DETECTION")
    results = model.predict(
        source=source,
        conf=0.5,
        save=True,
        show=True
    )
    return results

def run_video_tracking(model, source):
    """Run video tracking."""
    if not is_video_file(source):
        raise ValueError("❌ Tracking hanya mendukung file video. Masukkan file .mp4, .avi, .mov, .mkv, atau .webm")

    print("\n[MODE] VIDEO TRACKING")
    results = model.track(
        source=source,
        conf=0.7,
        iou=0.45,
        save=True,
        show=True,
        persist=True
    )
    print("✅ Tracking selesai. Hasil video disimpan di folder runs/detect/track")
    return results

# Main execution
try:
    # Load model
    model_path = "runs/detect/train-2/weights/best.pt"
    model = load_model(model_path)

    # Get user choice
    choice = get_user_choice()

    # Get file path
    source = get_file_path()

    # Execute based on choice
    if choice == "1":
        run_image_detection(model, source)
    elif choice == "2":
        run_video_tracking(model, source)
    else:
        print("❌ Pilihan tidak valid!")

except Exception as e:
    print(f"Error: {e}")


=== OBJECT DETECTION SYSTEM ===
1. Image Detection
2. Video Tracking


[MODE] VIDEO TRACKING

WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/364) e:\Vscode Projects\yolov11-project-final\datasets_test\videos\tank_video_2.mp4: 384x640 (no detections), 12.8ms
video 1/1 (frame 2/364) e:\Vscode Projects\yolov11-project-final\datasets_test\videos\tank_video_2.mp4: 384x640 (no detections), 11.9ms
video 1/1 (frame 3/364) e:\Vscode Projects\yolov11-project-final\datasets_test\videos\tank_vide